# ENVS 117 Week 3 Thursday: In-class Exercises
Work in pairs, about 10 minutes each. Run the setup cells first.

Name: Asha L.

## Setup

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

BASE = ("https://github.com/opengeos/"
        "datasets/releases/download/")
files = ["us/us_counties.geojson",
         "us/cities.csv",
         "raster/dem_90m.tif"]

Path("data").mkdir(exist_ok=True)
for f in files:
    name = f.split("/")[-1]
    if not Path("data/" + name).exists():      # skip if already downloaded
        urlretrieve(BASE + f, "data/" + name)
        print("downloaded", name)
    else:
        print("already have", name)

already have us_counties.geojson
already have cities.csv
already have dem_90m.tif


In [2]:
import os
import math
import pandas as pd
import geopandas as gpd
import rasterio

## T1. California counties, two formats

**1.** Read `data/us_counties.geojson` with `gpd.read_file`.

In [3]:
# your code here
counties  = gpd.read_file(
    "data/us_counties.geojson")


**2.** Print its type, shape, column names and CRS.

In [4]:
# your code here
print(counties.shape)
print(counties.crs)
print(type(counties))
print(counties.columns)

(3221, 7)
EPSG:4269
<class 'geopandas.geodataframe.GeoDataFrame'>
Index(['GEO_ID', 'STATE', 'COUNTY', 'NAME', 'LSAD', 'CENSUSAREA', 'geometry'], dtype='str')


**3.** Keep California: `STATE` equals `"06"`. How many counties?

In [5]:
# your code here
ca = counties[counties["STATE"] == "06"]
print(len(ca["COUNTY"].unique()))

58


**4.** What geometry type are they? What are the `total_bounds`?

In [6]:
# your code here
print(ca.geom_type.unique())
print(ca.total_bounds)

<StringArray>
['MultiPolygon']
Length: 1, dtype: str
[-124.409591     32.534156   -114.13442654   42.009518  ]


**5.** Save California as `data/ca_counties.gpkg` and as `data/ca_counties.shp`.

In [7]:
# your code here
ca.to_file("data/ca_counties.gpkg")
ca.to_file("data/ca_counties.shp")

**6.** List the data folder with `os.listdir`. How many files did each format create?

In [8]:
# your code here
print("Shapefile parts:", sorted(f for f in os.listdir("data") if f.startswith("ca_counties.") and not f.endswith(".gpkg")))
print("GeoPackage size:", round(os.path.getsize("data/ca_counties.gpkg") / 1e6, 2), "MB")

Shapefile parts: ['ca_counties.cpg', 'ca_counties.dbf', 'ca_counties.prj', 'ca_counties.shp', 'ca_counties.shx']
GeoPackage size: 0.74 MB


**Discuss:** Why is STATE stored as the text "06" and not the number 6? Which format would you email to a classmate?

This format allows for there to be storage and numbers easily read that are two digits, which helps for 50 states. I would most likely email the number 6 instead of 06 for conciseness.  

## T2. From a CSV of cities to points

**1.** Read `data/cities.csv` with `pd.read_csv`. Which columns hold the coordinates?

In [9]:
# your code here
cities = pd.read_csv("data/cities.csv")

print(cities)

#Latitude and Longitude have coordinates

                name sov_a3  latitude  longitude   pop_max     region
0     San Bernardino    USA  34.12038 -117.30003   1745000       West
1         Bridgeport    USA  41.17998  -73.19996   1018000  Northeast
2          Rochester    USA  43.17043  -77.61995    755000  Northeast
3           St. Paul    USA  44.94399  -93.08497    734854    Midwest
4           Billings    USA  45.78830 -108.54000    104552       West
..               ...    ...       ...        ...       ...        ...
104          Atlanta    USA  33.83001  -84.39995   4506000      South
105          Chicago    USA  41.82999  -87.75005   8990000    Midwest
106      Los Angeles    USA  33.98998 -118.17998  12500000       West
107  Washington D.C.    USA  38.89955  -77.00942   4338000      South
108         New York    USA  40.74998  -73.98002  19040000  Northeast

[109 rows x 6 columns]


**2.** Build a GeoDataFrame with `gpd.points_from_xy` and `crs="EPSG:4326"`. (x first, and x is longitude.)

In [10]:
# your code here
geo_df = gpd.GeoDataFrame(
    cities,
    geometry=gpd.points_from_xy(
        cities["longitude"], cities["latitude"]),
    crs="EPSG:4326")

**3.** Is its CRS the same as the counties' CRS? Print the comparison.

In [11]:
# your code here
print(geo_df.crs)
print(counties.crs)
print(geo_df.crs == counties.crs)

EPSG:4326
EPSG:4269
False


**4.** Convert the points to the counties' CRS with `to_crs`.

In [12]:
# your code here
geo_df = geo_df.to_crs(counties.crs)

**5.** Pick "California" cities with a box: latitude 32.5 to 42.0 and longitude -124.5 to -114.1. Use `.between()`.

In [13]:
# your code here
ca_cities = geo_df[
    geo_df["latitude"].between(32.5, 42.0) &
    geo_df["longitude"].between(-124.5, -114.1)
]
print(ca_cities)

               name sov_a3  latitude  longitude   pop_max region  \
0    San Bernardino    USA  34.12038 -117.30003   1745000   West   
15    Santa Barbara    USA  34.43499 -119.71999    181632   West   
16           Fresno    USA  36.74772 -119.77298    616353   West   
17           Eureka    USA  40.80222 -124.14750     42398   West   
19             Reno    USA  39.52998 -119.82001    328585   West   
20             Elko    USA  40.83251 -115.76199     19252   West   
56         San Jose    USA  37.29998 -121.84999   1668000   West   
57       Sacramento    USA  38.57502 -121.47004   1604000   West   
58        Las Vegas    USA  36.21000 -115.22001   1823000   West   
91        San Diego    USA  32.82002 -117.17999   2916000   West   
100   San Francisco    USA  37.74001 -122.45998   3450000   West   
106     Los Angeles    USA  33.98998 -118.17998  12500000   West   

                        geometry  
0    POINT (-117.30003 34.12038)  
15   POINT (-119.71999 34.43499)  
16   POINT

**Look closely:** three cities in your box are not in California. Which ones, and why did the box catch them? Reno, Elko, and Las Vegas are all in Nevada. They were not caught because their latitude is between the defined latitudes and their longitude is between the defined longitudes. This is why are they are included. 

## T3. The Sierra Nevada elevation model

**1-2.** Open `data/dem_90m.tif` with rasterio inside a `with` block. Print crs, width, height, count, dtypes, res, nodata and bounds.

**3.** Read band 1 into `dem`. Print its shape and type.

In [14]:
# your code here
with rasterio.open("data/dem_90m.tif") as src:
    print(src.crs)
    print(src.res)
    print(src.count)
    print(src.dtypes)
    print(src.width)
    print(src.height)
    print(src.nodata)
    print(src.bounds)
    dem = src.read(1)

dem.shape
type(dem)

EPSG:3857
(90.0, 89.99579177642138)
1
('int16',)
4269
3113
None
BoundingBox(left=-13442488.3428, bottom=4388214.6777, right=-13058278.3428, top=4668371.5775)


numpy.ndarray

**4.** Find the lowest, highest and mean elevation, rounded to 1 decimal.

In [15]:
# your code here
print(round(dem.max(), 1))
print(round(dem.min(), 1))
print(round(dem.mean(), 1))

4338
15
1582.3


**5.** How many cells are above 3,000 m? Hint: `(dem > 3000).sum()`

In [16]:
# your code here
print((dem > 3000).sum())

859074


**Think:** the resolution says 90 m, but the CRS is Web Mercator. At 37.6 N, how wide is one cell on the ground?

In [17]:
# your code here
santa_clara = ca[ca["NAME"] == "Santa Clara"]
lat = 37.6
res = 90
print("EPSG:3857 (Mercator): ", round(santa_clara.to_crs(3857).area.iloc[0] / 1e6), "km2")


EPSG:3857 (Mercator):  5338 km2


## T4. Santa Clara County, three areas

**1.** From your California counties, select `NAME` equal to `"Santa Clara"`.

In [18]:
# your code here
santa_clara = ca[ca["NAME"] == "Santa Clara"]
print(santa_clara)

             GEO_ID STATE COUNTY         NAME    LSAD  CENSUSAREA  \
235  0500000US06085    06    085  Santa Clara  County      1290.1   

                                              geometry  
235  MULTIPOLYGON (((-121.99727 37.15208, -121.9980...  


**2.** Print `.area` in its current CRS. Read the warning.

In [19]:
# your code here
print(santa_clara.area)

235    0.342953
dtype: float64


/var/folders/zs/4zds_td97pl9hxx9pcf2mt2c0000gn/T/ipykernel_74829/677424572.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print(santa_clara.area)


**3.** Convert to EPSG:3310 and to EPSG:3857. Print each area in km².

In [20]:
# your code here
# your code here
sc_epsg3310 = round(santa_clara.to_crs(3310).area.iloc[0] / 1e6)
sc_epsg_3857 = round(santa_clara.to_crs(3857).area.iloc[0] / 1e6)
print("EPSG:3310 (Albers):   ",sc_epsg3310, "km2")
print("EPSG:3857 (Mercator): ", sc_epsg_3857, "km2")

EPSG:3310 (Albers):    3378 km2
EPSG:3857 (Mercator):  5338 km2


**4.** Compare with `CENSUSAREA`, the Census land area in square miles (1 sq mi = 2.59 km²).

In [21]:
# your code here
print("Census land area:", round(santa_clara["CENSUSAREA"].iloc[0] * 2.59), "km2")

#The Census land area is fewer km compared to albers and mercator.

Census land area: 3341 km2


**Write two sentences:** Which number would you put in a report, and why? Why is the Web Mercator area so much larger?

Web Mercator is primarily designed for webmaps, and it is larger because it appears to stretch numbers. I would use Albers for a report because it works best for selecting California, or perhaps the Census land area if that is used. 

## Stretch: start Step 1
Open one dataset for your own project and print its CRS and size.

In [26]:
# your project data here
folder = "prism_ppt_us_30s_202606"
file = "prism_ppt_us_30s_202606.tif"
file_path = os.path.join(folder, file)
with rasterio.open("data/prism_ppt_us_30s_202606/prism_ppt_us_30s_202606.tif") as src:
    print(src.crs)
    print(src.height)
    print(src.width)
    print(src.bounds)

EPSG:4269
3105
7025
BoundingBox(left=-125.02083333333351, bottom=24.062500001034504, right=-66.47916666900852, top=49.9374999999995)
